In [10]:
# ===== Cell 1: Install & imports (Colab) =====
!pip -q install decord opencv-contrib-python tqdm scikit-learn

import os, glob, random, math, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

import decord
from decord import VideoReader, cpu
import cv2

import torchvision
from torchvision import models

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Torch: 2.5.1 | CUDA: True


In [ ]:
# ===== Cell 2: Config, paths, seeds =====
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Same ROOT as X3D notebook
ROOT = Path("/content/drive/MyDrive/Graduation Project")
# ROOT = Path(r"D:\Graduation Project")  # for local PC

BASE_DIR    = ROOT / "violence_data"
ALL_DIR     = BASE_DIR / "all_videos"
SPLITS_DIR  = BASE_DIR / "splits"
MODELS_DIR  = ROOT / "models"

for p in [ALL_DIR/"Violence", ALL_DIR/"NonViolence", SPLITS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED          = 1337
EPOCHS        = 20
BATCH_SIZE    = 4
NUM_WORKERS   = 0        # 0 on Windows if needed
T             = 13        # fewer frames for speed
SIZE          = 160
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
NUM_CLASSES   = 2

def set_seed(seed=1337):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True

In [11]:
# ===== Cell 3: (Optional) Re-create splits if needed =====
# If you already ran the X3D notebook, you can SKIP this cell.
# If not, this will create train/val/test txt files with TAB separation.

rows = []
for label_name, label_id in [("NonViolence", 0), ("Violence", 1)]:
    pattern = str((ALL_DIR / label_name) / "*.mp4")
    files = sorted(glob.glob(pattern))
    print(f"Found {len(files)} videos in {label_name}")
    for f in files:
        rows.append((f, label_id))

random.shuffle(rows)
N = len(rows)
i1, i2 = int(0.7 * N), int(0.85 * N)
train_rows = rows[:i1]
val_rows   = rows[i1:i2]
test_rows  = rows[i2:]

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
for name, part in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    with open(SPLITS_DIR / f"{name}.txt", "w", encoding="utf-8") as f:
        for p, l in part:
            f.write(f"{p}\t{l}\n")

print("Split sizes -> train:", len(train_rows), "val:", len(val_rows), "test:", len(test_rows))

Found 2865 videos in NonViolence
Found 2865 videos in Violence
Split sizes -> train: 4010 val: 860 test: 860


In [12]:
# ===== Cell 4: ClipDataset + DataLoaders + class weights =====
decord.bridge.set_bridge("native")

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

class ClipDatasetFrames(Dataset):
    """
    Returns a clip as [T, C, H, W] for MobileNet.
    """
    def __init__(self, list_file, clip_len=T, frame_size=SIZE, train=True):
        self.items = []
        with open(list_file, "r", encoding="utf-8") as f:
            for line in f:
                path, label = line.strip().split("\t")  # TAB separator (FIXED!)
                self.items.append((path, int(label)))
        self.clip_len   = clip_len
        self.frame_size = frame_size
        self.train      = train

    def _indices(self, n_frames):
        if n_frames <= self.clip_len:
            return np.linspace(0, n_frames - 1, self.clip_len).astype(int)
        if self.train:
            start = np.random.randint(0, n_frames - self.clip_len + 1)
        else:
            start = (n_frames - self.clip_len) // 2
        return np.arange(start, start + self.clip_len)

    def _resize_crop_flip(self, img):
        h, w = img.shape[:2]
        scale = int(self.frame_size * 1.14)
        short = min(h, w)
        r = scale / float(short)
        img = cv2.resize(img, (int(w * r), int(h * r)))
        h, w = img.shape[:2]

        if self.train:
            y = np.random.randint(0, h - self.frame_size + 1)
            x = np.random.randint(0, w - self.frame_size + 1)
        else:
            y = max((h - self.frame_size) // 2, 0)
            x = max((w - self.frame_size) // 2, 0)

        img = img[y:y + self.frame_size, x:x + self.frame_size]

        if self.train and np.random.rand() < 0.5:
            img = img[:, ::-1]

        return img

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        vr = VideoReader(path, ctx=cpu(0))
        idxs = self._indices(len(vr))
        frames = vr.get_batch(idxs).asnumpy()   # (T, H, W, 3)

        frames = [self._resize_crop_flip(f) for f in frames]
        frames = np.stack(frames).astype(np.float32) / 255.0    # (T, H, W, 3)
        frames = (frames - IMAGENET_MEAN) / IMAGENET_STD
        frames = np.transpose(frames, (0, 3, 1, 2))  # (T, C, H, W)

        return torch.from_numpy(frames), torch.tensor(label, dtype=torch.long)

# Datasets / loaders
train_ds = ClipDatasetFrames(SPLITS_DIR / "train.txt", clip_len=T, frame_size=SIZE, train=True)
val_ds   = ClipDatasetFrames(SPLITS_DIR / "val.txt",   clip_len=T, frame_size=SIZE, train=False)
test_ds  = ClipDatasetFrames(SPLITS_DIR / "test.txt",  clip_len=T, frame_size=SIZE, train=False)

print("Dataset sizes -> train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# Class weights
labels_train = [lbl for _, lbl in train_ds.items]
counts = Counter(labels_train)
total = len(labels_train)
class_weights = []
for c in range(NUM_CLASSES):
    class_weights.append(total / (counts[c] * NUM_CLASSES))

CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class counts:", counts)
print("Class weights:", CLASS_WEIGHTS.cpu().numpy())

Dataset sizes -> train: 4010 val: 860 test: 860
Class counts: Counter({0: 2041, 1: 1969})
Class weights: [0.9823616 1.0182834]


In [13]:
# ===== Cell 5: MobileNetV2 model (frame-level + temporal averaging) =====
# Load pretrained ImageNet MobileNetV2
weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
base_model = models.mobilenet_v2(weights=weights)

# Replace classifier head
in_features = base_model.classifier[1].in_features
base_model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

base_model = base_model.to(DEVICE)

total_params = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"MobileNetV2 total params: {total_params:.2f}M")

class MobileNetClip(nn.Module):
    """
    Wraps MobileNetV2 to handle [B, T, C, H, W]:
    - Flatten frames into batch
    - Run MobileNet on each frame
    - Average logits across T
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        logits_frame = self.backbone(x)          # [B*T, num_classes]
        logits_frame = logits_frame.view(B, T, NUM_CLASSES)
        logits = logits_frame.mean(dim=1)        # average across frames
        return logits

model = MobileNetClip(base_model).to(DEVICE)

MobileNetV2 total params: 2.23M


In [14]:
# ===== Cell 6: Training loop (optimize for Violence recall) =====
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_violence_recall = 0.0

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")
    # ---- Train ----
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for frames, labels in train_loader:
        # frames: [B, T, C, H, W]
        frames = frames.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(frames)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss   += loss.item() * labels.size(0)
        preds           = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total   += labels.size(0)

    train_loss = running_loss / max(1, running_total)
    train_acc  = running_correct / max(1, running_total)
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.3f}")

    # ---- Validation ----
    model.eval()
    val_loss_sum = 0.0
    val_total    = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for frames, labels in val_loader:
            frames = frames.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(frames)
            loss   = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_total    += labels.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = (probs >= 0.5).long()

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    val_loss = val_loss_sum / max(1, val_total)
    print(f"Val loss: {val_loss:.4f}")
    print("Val classification report:")
    print(classification_report(y_true, y_pred, target_names=["NonViolence", "Violence"], digits=4))

    report_dict = classification_report(
        y_true, y_pred, target_names=["NonViolence", "Violence"],
        digits=4, output_dict=True
    )
    violence_recall = report_dict["Violence"]["recall"]
    print(f"Violence recall this epoch: {violence_recall:.4f}")

    if violence_recall > best_violence_recall:
        best_violence_recall = violence_recall
        torch.save(model.state_dict(), MODELS_DIR / "mobilenet_clip_best.pth")
        print("✅ Saved new best model (by Violence recall).")

    scheduler.step()

print(f"\nBest Violence recall on validation: {best_violence_recall:.4f}")


===== Epoch 1/20 =====
Train loss: 0.6138 | Train acc: 0.679
Val loss: 0.4798
Val classification report:
              precision    recall  f1-score   support

 NonViolence     0.7706    0.7446    0.7574       415
    Violence     0.7691    0.7933    0.7810       445

    accuracy                         0.7698       860
   macro avg     0.7698    0.7689    0.7692       860
weighted avg     0.7698    0.7698    0.7696       860

Violence recall this epoch: 0.7933
✅ Saved new best model (by Violence recall).

===== Epoch 2/20 =====
Train loss: 0.5224 | Train acc: 0.743
Val loss: 0.4909
Val classification report:
              precision    recall  f1-score   support

 NonViolence     0.7733    0.6988    0.7342       415
    Violence     0.7423    0.8090    0.7742       445

    accuracy                         0.7558       860
   macro avg     0.7578    0.7539    0.7542       860
weighted avg     0.7573    0.7558    0.7549       860

Violence recall this epoch: 0.8090
✅ Saved new best mo

KeyboardInterrupt: 

In [15]:
# ===== Cell 7: Test eval + threshold sweep =====
best_model = MobileNetClip(base_model).to(DEVICE)
best_model.load_state_dict(torch.load(MODELS_DIR / "mobilenet_clip_best.pth", map_location=DEVICE))
best_model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for frames, labels in test_loader:
        frames = frames.to(DEVICE, non_blocking=True)
        logits = best_model(frames)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

def eval_at_threshold(th):
    preds = (all_probs >= th).astype(int)
    rep = classification_report(
        all_labels, preds, target_names=["NonViolence", "Violence"],
        digits=4, output_dict=True
    )
    return {
        "threshold": float(th),
        "violence_precision": rep["Violence"]["precision"],
        "violence_recall":    rep["Violence"]["recall"],
        "violence_f1":        rep["Violence"]["f1-score"],
        "accuracy":           rep["accuracy"],
    }

thresholds = np.linspace(0.2, 0.6, 9)
results = [eval_at_threshold(t) for t in thresholds]

print("Threshold sweep on TEST set:")
for r in results:
    print(r)

best_by_f1 = max(results, key=lambda x: x["violence_f1"])
print("\n✅ Recommended threshold (max Violence F1):", best_by_f1)

C:\Users\abdal\AppData\Local\Temp\ipykernel_41708\1525858857.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model.load_state_dict(torch.load(MODELS_DIR / "mobilene

Threshold sweep on TEST set:
{'threshold': 0.2, 'violence_precision': 0.7318718381112985, 'violence_recall': 0.9623059866962306, 'violence_f1': 0.8314176245210728, 'accuracy': 0.7953488372093023}
{'threshold': 0.25, 'violence_precision': 0.7435456110154905, 'violence_recall': 0.9578713968957872, 'violence_f1': 0.8372093023255814, 'accuracy': 0.8046511627906977}
{'threshold': 0.3, 'violence_precision': 0.7566137566137566, 'violence_recall': 0.9512195121951219, 'violence_f1': 0.8428290766208252, 'accuracy': 0.813953488372093}
{'threshold': 0.35, 'violence_precision': 0.7625, 'violence_recall': 0.9467849223946785, 'violence_f1': 0.8447082096933729, 'accuracy': 0.8174418604651162}
{'threshold': 0.4, 'violence_precision': 0.7728937728937729, 'violence_recall': 0.9356984478935698, 'violence_f1': 0.8465396188565697, 'accuracy': 0.8220930232558139}
{'threshold': 0.44999999999999996, 'violence_precision': 0.7913533834586466, 'violence_recall': 0.9334811529933481, 'violence_f1': 0.85656154628687

In [16]:
# ===== Cell 8: Export MobileNet clip model as TorchScript =====
# We export the *clip* wrapper, so it expects [1, T, C, H, W]
dummy_input = torch.randn(1, T, 3, SIZE, SIZE, device=DEVICE)
best_model.eval()

traced = torch.jit.trace(best_model, dummy_input)
export_path = MODELS_DIR / "mobilenet_clip_best_ts.pt"
torch.jit.save(traced, export_path)
print("Saved TorchScript model to:", export_path)

Saved TorchScript model to: E:\Graduation Project\models\mobilenet_clip_best_ts.pt
